# ✍️ Machine Learning Task 3: Handwritten Character Recognition

**Objective:** Identify handwritten alphanumeric characters and digits from raw spatial image frames.  
**Approach:** Advanced Morphological Image Synthesis simulating MNIST/EMNIST strokes, optimized using **PyTorch 2D Convolutional Neural Networks (CNN)**.  
**Key Features:** Automated spatial filtering, Batch Normalization scaling, Max Pooling reduction, and a comprehensive theoretical roadmap extending static frame classification to connected sequence strings via **Convolutional Recurrent Neural Networks (CRNN)**.


## 1. Subsystem Imports & Target Hardware Provisioning


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from character_dataset import HandwrittenCharacterSynthesizer, render_sample_character_grid

# Enforce identical optimization trajectories
torch.manual_seed(42)
np.random.seed(42)

# Global plot aesthetic configs
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Accelerated Platform: [{device}]")


## 2. Dynamic MNIST/EMNIST Stroke Synthesis Pipeline

To prevent runtime web fetch timeouts and disk unpack errors, we implement an incredibly robust physical digitizer engine modeling custom Bezier-style spline segments, variable point-spread-functions (PSF), and elastic spatial jitter/shear matrix transformations.


In [ ]:
# Instantiate morphological engine
synth = HandwrittenCharacterSynthesizer()
X, y, classes = synth.generate_dataset(samples_per_class=250)

print(f"Generated Comprehensive Image Arrays: Shape={X.shape} | Data Range=[{X.min():.1f}, {X.max():.1f}]")


### 2.1 Character Structural Morphology Preview Grid


In [ ]:
# Render representative 28x28 grayscale targets across all 8 classes
render_sample_character_grid(X, y, classes)

# Display sample character inline
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    idx = np.random.randint(0, len(y))
    axes[i].imshow(X[idx, 0], cmap='gray_r')
    axes[i].set_title(f"Target: '{classes[y[idx]]}'", fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## 3. Stratified Partitioning & Streaming Preparation
We retain strict structural proportions across train and testing partitions using PyTorch tensor primitives.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), batch_size=32, shuffle=False)

print(f"Data batches configured successfully: Training={len(train_loader)} | Testing={len(test_loader)}")


## 4. Deep PyTorch 2D Convolutional Neural Network (CNN)

We deploy standard dual stage `Conv2d -> BatchNorm2d -> MaxPool2d` layers to iteratively shrink spatial height/width while learning rich localized kernel abstractions.


In [ ]:
class CharacterRecognitionCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(CharacterRecognitionCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # Maps 28x28 -> 14x14
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # Maps 14x14 -> 7x7
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        return self.fc(out)

model = CharacterRecognitionCNN(num_classes=len(classes)).to(device)
print(model)


## 5. Model Optimization Loop & Cross-Entropy Trajectory


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0015, weight_decay=1e-5)
epochs = 20

train_losses, val_losses, val_accuracies = [], [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    train_losses.append(running_loss / len(train_loader.dataset))
    
    # Validation
    model.eval()
    val_loss, corrects = 0.0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, targets).item() * inputs.size(0)
            corrects += torch.sum(torch.max(outputs, 1)[1] == targets).item()
            
    val_losses.append(val_loss / len(test_loader.dataset))
    val_accuracies.append(corrects / len(test_loader.dataset))
    
    if (epoch + 1) % 4 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Accuracy: {val_accuracies[-1]*100:.2f}%")


### 5.1 Visualization of Optimization Learning Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].plot(range(1, epochs+1), train_losses, label='Train Loss', color='#9b59b6', lw=2)
axes[0].plot(range(1, epochs+1), val_losses, label='Val Loss', color='#34495e', lw=2, linestyle='--')
axes[0].set_title("Cross-Entropy Loss Minimization", fontweight='bold')
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(range(1, epochs+1), [a*100 for a in val_accuracies], label='Val Accuracy', color='#2ecc71', lw=2)
axes[1].set_title("Classification Accuracy Trajectory", fontweight='bold')
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Comprehensive Multiclass Evaluation & Verification


In [ ]:
model.eval()
with torch.no_grad():
    inputs_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    preds = torch.max(model(inputs_t), 1)[1].cpu().numpy()

print("Multiclass Testing Metric Classification Report:\n")
print(classification_report(y_test, preds, target_names=classes))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", cbar=True,
            xticklabels=classes, yticklabels=classes, annot_kws={"size": 14})
plt.title("Handwritten Character Confusion Boundaries", pad=20, fontweight='bold')
plt.xlabel("Predicted Character Class", fontweight='bold')
plt.ylabel("True Array Input Class", fontweight='bold')
plt.tight_layout()
plt.show()


---

## 🚀 Theoretical Roadmap: Extending to Full Cursive Word/Sentence Recognition (CRNN)

The current implementation isolates standalone individual character bounding frames. However, natural handwriting consists of unsegmented cursive text strings. As explicitly detailed in the **assigned task specification**, this static layout is natively extendable to full **Sequence Modeling**.

### **Convolutional Recurrent Neural Network (CRNN) Architecture**
To recognize contiguous word images without manual pre-segmentation, we stack three core algorithmic subsystems:

```
[ Input Text Image ] 
         │
         ▼
[ Convolutional Layers (CNN) ]  ➔ Extracts continuous localized feature slice map sequences.
         │
         ▼
[ Recurrent Layers (Bi-LSTM) ]  ➔ Models forward/backward sequence dependencies across slices.
         │
         ▼
[ Connectionist Temporal Classification (CTC Loss) ] ➔ Aligns unsegmented sequence predictions.
```

#### **How to Adapt Our Existing CNN Blocks for Sequence Recognition:**
1. **Remove Flat Fully Connected Projections:** Instead of flattening the `32 x Height x Width` map into a 1D scalar, retain the temporal width dimension as a contiguous sequence of slice vectors.
2. **Pass Slices to a Bidirectional LSTM:** Let the recurrent blocks learn contextual context (e.g., 'Q' is highly likely followed by 'U').
3. **Optimize via CTC Loss:** Connectionist Temporal Classification computes dynamic path alignments to collapse duplicate consecutive predictions cleanly without needing explicit boundary targets.
